# IOR Read/Write – Per-Run Analysis

Combines all per-run CSV files in this directory into a single DataFrame,
extracts `n` (number of nodes) from the filename, counts config occurrences,
and computes the coefficient of variation (CV%) for read/write bandwidth.

In [1]:
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

## 1. Load and combine all CSVs

Filename convention: `ior_rw_n-{N}_ppn-{PPN}_tx-{TX}_{timestamp}.csv`

The `n`, `ppn`, and `tx` parameters are extracted from the filename and appended as columns.

In [2]:
data_dir = Path(".")  # notebook lives alongside the CSVs
csv_files = sorted(data_dir.glob("ior_rw_n-*.csv"))
print(f"Found {len(csv_files)} CSV files")

frames = []
for f in csv_files:
    m = re.search(r'_n-(\d+)_ppn-(\d+)_tx-(\w+)_', f.stem)
    if not m:
        print(f"  skipping (no match): {f.name}")
        continue
    df = pd.read_csv(f)
    df.columns = df.columns.str.strip()
    df["n"]        = int(m.group(1))
    df["ppn"]      = int(m.group(2))
    df["tx"]       = m.group(3).upper()   # normalise: 2M, 16M
    df["filename"] = f.name
    frames.append(df)

combined = pd.concat(frames, ignore_index=True)
combined["access"]   = combined["access"].str.strip().str.lower()
combined["bw_GiB_s"] = combined["bw(MiB/s)"] / 1024.0

print(f"Total rows : {len(combined)}")
combined.head()

Found 24 CSV files
Total rows : 472


,access,bw(MiB/s),IOPS,Latency,block(KiB),xfer(KiB),open(s),wr/rd(s),close(s),total(s),numTasks,iter,n,ppn,tx,filename,bw_GiB_s
0,write,127578.8913,8086.2273,0.0295,131072.0,16384.0,0.0069,0.2533,0.0175,0.2568,16,0,1,16,16M,ior_rw_n-1_ppn-16_tx-16m_1782525728.csv,124.588761
1,read,103878.9794,6560.8881,0.0366,131072.0,16384.0,0.0066,0.3122,0.0195,0.3154,16,0,1,16,16M,ior_rw_n-1_ppn-16_tx-16m_1782525728.csv,101.444316
2,write,129122.7996,8171.6367,0.0302,131072.0,16384.0,0.0062,0.2506,0.0120,0.2538,16,1,1,16,16M,ior_rw_n-1_ppn-16_tx-16m_1782525728.csv,126.096484
3,read,104069.0155,6568.0166,0.0380,131072.0,16384.0,0.0062,0.3118,0.0184,0.3149,16,1,1,16,16M,ior_rw_n-1_ppn-16_tx-16m_1782525728.csv,101.629898
4,write,130488.9799,8264.4317,0.0303,131072.0,16384.0,0.0065,0.2478,0.0061,0.2511,16,2,1,16,16M,ior_rw_n-1_ppn-16_tx-16m_1782525728.csv,127.430644


In [3]:
out_path = data_dir / "ior_rw_combined.csv"
combined.to_csv(out_path, index=False)
print(f"Saved combined CSV → {out_path.resolve()}")

Saved combined CSV → /Users/xmei/Documents/LDRD/pdsw26_daos-bench-res/results/aurora/ior/ior-rw/ior_rw_combined.csv


## 2. Config counts

Each row in a CSV = one iter. Count how many iters exist per `(n, ppn, tx, access)` config.

In [ ]:
config_cols = ["n", "ppn", "tx"]

iter_counts = (
    combined.groupby(config_cols + ["access"])
    .size()
    .rename("num_iters")
    .reset_index()
    .sort_values(config_cols + ["access"])
    .reset_index(drop=True)
)
display(iter_counts)

## 3. CV% for Read / Write bandwidth

CV (coefficient of variation) = σ / μ × 100 %, computed **across all iters** (rows) per config.  
Each row in the CSV is one iter measurement.

In [ ]:
def cv_pct(x):
    return x.std(ddof=1) / x.mean() * 100 if len(x) > 1 and x.mean() != 0 else np.nan

stats = (
    combined.groupby(config_cols + ["access"])["bw_GiB_s"]
    .agg(mean_GiB_s="mean", std_GiB_s="std", cv_pct=cv_pct, num_iters="count")
    .reset_index()
    .sort_values(config_cols + ["access"])
)

CV_THRESH = 5.0

def highlight_high_cv(row):
    color = "background-color: #ffe0e0" if pd.notna(row["cv_pct"]) and row["cv_pct"] > CV_THRESH else ""
    return [color] * len(row)

display(
    stats.style
    .apply(highlight_high_cv, axis=1)
    .format({"mean_GiB_s": "{:.3f}", "std_GiB_s": "{:.3f}", "cv_pct": "{:.2f}"}, na_rep="—")
    .set_caption(f"CV% across iters per config × access  (red = CV > {CV_THRESH}%)")
)

### CV% bar chart

In [ ]:
sns.set_theme(style="ticks")
mpl.rcParams.update({
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.color":        "#e0e0e0",
    "grid.linewidth":    0.8,
    "axes.facecolor":    "#fafafa",
    "figure.facecolor":  "white",
    "font.size":         11,
})

for access_type in ["write", "read"]:
    sub = stats[stats["access"] == access_type].copy()
    sub["config"] = sub.apply(
        lambda r: f"n={int(r.n)} ppn={int(r.ppn)} tx={r.tx}", axis=1
    )
    sub = sub.sort_values(["n", "ppn", "tx"])

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ["#d62728" if v > CV_THRESH else "#1f77b4" for v in sub["cv_pct"]]
    ax.bar(sub["config"], sub["cv_pct"], color=colors, edgecolor="none")
    ax.axhline(CV_THRESH, color="#d62728", linewidth=1, linestyle="--", label=f"CV = {CV_THRESH}%")
    ax.set_xlabel("Config (n, ppn, tx)")
    ax.set_ylabel("CV (%)")
    ax.set_title(f"{access_type.capitalize()} BW CV% across iters")
    ax.legend()
    plt.xticks(rotation=45, ha="right", fontsize=9)
    fig.tight_layout()
    plt.show()

### BW distribution across iters (box plot)

In [ ]:
combined["config"] = combined.apply(
    lambda r: f"n={int(r.n)} ppn={int(r.ppn)} tx={r.tx}", axis=1
)
config_order = (
    combined[["n", "ppn", "tx", "config"]]
    .drop_duplicates()
    .sort_values(["n", "ppn", "tx"])["config"]
    .tolist()
)

for access_type in ["write", "read"]:
    sub = combined[combined["access"] == access_type]
    valid_order = [c for c in config_order if c in sub["config"].values]

    fig, ax = plt.subplots(figsize=(12, 4))
    sns.boxplot(
        data=sub, x="config", y="bw_GiB_s",
        order=valid_order, palette="Blues", ax=ax, linewidth=0.8
    )
    ax.set_xlabel("Config (n, ppn, tx)")
    ax.set_ylabel("Bandwidth (GiB/s)")
    ax.set_title(f"{access_type.capitalize()} BW distribution across iters")
    plt.xticks(rotation=45, ha="right", fontsize=9)
    fig.tight_layout()
    plt.show()

## 4. Peak Read/Write BW for n=1 and n=32

In [ ]:
peak_rows = []
for n_val in [1, 32]:
    for access_type in ["write", "read"]:
        sub = combined[(combined["n"] == n_val) & (combined["access"] == access_type)]
        idx = sub["bw_GiB_s"].idxmax()
        row = sub.loc[idx]
        peak_rows.append({
            "n":          n_val,
            "access":     access_type,
            "peak_BW_GiB_s": row["bw_GiB_s"],
            "ppn":        int(row["ppn"]),
            "tx":         row["tx"],
            "iter":       int(row["iter"]),
            "filename":   row["filename"],
        })

peak_df = pd.DataFrame(peak_rows)
display(
    peak_df.style
    .format({"peak_BW_GiB_s": "{:.3f}"})
    .set_caption("Peak BW per n × access (best single iter across all configs)")
    .hide(axis="index")
)